In [ ]:
# imports
import copy
import cv2
import glob
import matplotlib.pyplot as plt
import numpy as np
import os
import open3d as o3d
from open3d.web_visualizer import draw as web_draw
import sys
from ultralytics import YOLO

sys.path.append(os.path.abspath('../src'))
from camera import Camera
from evaluation import Evaluation, compare_evaluations
from paths import PathManager
from yolo import predict_segment

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[Open3D INFO] Resetting default logger to print to terminal.


ModuleNotFoundError: No module named 'yolo11'

In [ ]:
# Settings for this notebook
apple_index = 0
web_visualization = False
searched_item = 'apple'

In [ ]:
# draw function: depending on web_visualization
def draw(point_clouds):
    global web_visualization
    if web_visualization:
        web_draw(point_clouds)
    else:
        o3d.visualization.draw_geometries(point_clouds)

In [ ]:
pm = PathManager(searched_item)
rgb_image = np.load(pm.get_rgb_image_path(apple_index))
depth_image = np.load(pm.get_depth_image_path(apple_index))

In [ ]:
pts, seg_image = predict_segment(rgb_image, searched_item)

In [ ]:
# Create a mask initialized with zeros (same shape as depth image)
mask = np.zeros_like(depth_image)
cv2.fillPoly(mask, [pts], color=1)  # Set inside polygon to 1
masked_depth_image = depth_image * mask

In [ ]:
cam = Camera.from_yaml("../data/camera_intrinsics.yaml", "D435i")

In [ ]:
def get_scene_point_cloud(depth_img, d_width, d_height, cam_params):
    (fx, fy, cx, cy) = cam_params
    depth_img = o3d.geometry.Image(depth_img)
    cam_intrinsics = o3d.camera.PinholeCameraIntrinsic(
        width=d_width,
        height=d_height,
        fx=fx,
        fy=fy,
        cx=cx,
        cy=cy)
    pcd = o3d.geometry.PointCloud.create_from_depth_image(
        depth=depth_img,
        intrinsic=cam_intrinsics,
        extrinsic=np.eye(4)
    )
    return pcd

In [ ]:
pcd = get_scene_point_cloud(depth_image, cam.width, cam.height, cam.get_camera_intrinsics())
cropped_pcd = get_scene_point_cloud(masked_depth_image, cam.width, cam.height, cam.get_camera_intrinsics())

In [ ]:
position = cropped_pcd.get_center()

In [ ]:
model_pcd = o3d.io.read_point_cloud(f"../data/{searched_item}/{searched_item}.ply")

In [ ]:
def get_templates(searched_item):
    template_dir = f"../data/{searched_item}"
    file_names = f"{searched_item}_template_*.ply"
    template_files = sorted(glob.glob(os.path.join(template_dir, file_names)))

    templates = []
    for path in template_files:
        pcd = o3d.io.read_point_cloud(path)
        templates.append(pcd)
    return templates

In [ ]:
templates = get_templates(searched_item)
print(templates)

In [ ]:
def prepare_point_cloud(pcd):
    filtered_pcd = copy.deepcopy(pcd)

    # remove outliers
    filtered_pcd, ind = filtered_pcd.remove_statistical_outlier(nb_neighbors=50, std_ratio=0.7)

    # estimate normals
    filtered_pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))

    return filtered_pcd

In [ ]:
def preprocess_point_cloud(pcd, voxel_size):
    pcd_down = pcd.voxel_down_sample(voxel_size)
    pcd_down.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*2, max_nn=30)
    )
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*5, max_nn=100)
    )
    return pcd_down, fpfh

In [ ]:
filtered_scene = prepare_point_cloud(cropped_pcd)

In [ ]:
evals = []
for model in templates:
    T = np.eye(4)
    T[:3, 3] = filtered_scene.get_center() - model.get_center()

    # transform with calculatet T
    transformed_model = copy.deepcopy(model).transform(T)

    voxel_size = 0.005 # in meters -> 5mm
    scene_down, scene_fpfh = preprocess_point_cloud(filtered_scene, voxel_size)
    model_down, model_fpfh = preprocess_point_cloud(transformed_model, voxel_size)


    result_ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        model_down,             # source
        scene_down,             # target
        model_fpfh,             # source feature
        scene_fpfh,             # target feature
        mutual_filter=False,    # mutual_filter
        max_correspondence_distance=voxel_size*1.5,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=4,
        checkers=[
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(voxel_size*1.5),
        ],
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 500)
    )

    source = copy.deepcopy(model_down)
    target = copy.deepcopy(scene_down)
    threshold = voxel_size

    reg_p2p = o3d.pipelines.registration.registration_icp(
        source=source,
        target=target,
        max_correspondence_distance=threshold,
        init=result_ransac.transformation,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=100),
    )

    transformed_source = copy.deepcopy(source).transform(reg_p2p.transformation)

    eval = Evaluation(source, target, threshold, reg_p2p.transformation, T)
    print(f"{eval} \n")
    evals.append(eval)


In [ ]:
ev = compare_evaluations(evals, sort_by="fitness", reverse=True, top_n=1, verbose=False)
print(ev[0])